In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model(model="gemini-3.1-flash-lite", model_provider="google-genai")
standard_model = init_chat_model(model="gemma4", model_provider="openai", api_key="dummy", base_url="http://localhost:8080/v1")


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model=standard_model,
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

Oh, good morning! 👋

Let me check my schedule/notes... Hmm, I think I was planning to water the office plant sometime this afternoon. I haven't gotten to it yet, but I will make sure to take care of it right away! 🌿

Is there anything else I can help you with while I'm at it? Maybe grab you a coffee? 😊


In [5]:
print(response["messages"][-1].response_metadata["model_name"])

gemma-4-E4B-it-UD-Q4_K_XL.gguf


In [6]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': "I checked the roots earlier when I was wiping down the leaves, and they aren't poking out of the drainage holes yet. Based on its current growth rate, I’d say we probably won't need to repot it until the start of next season—maybe in about three months?\n\nI'll keep a close eye on it, though. If it starts getting root-bound sooner than that, I’ll let you know immediately! Do you want me to add a reminder to the office calendar for when we should pick up a larger pot?", 'extras': {'signature': 'EnEKbwERTTIP25qTIu/ksoH7H2nkWe5tMikUJQlGGk00EPCAoTvGh5FWH1YjIR3KkeSsT66yQNveYrFkvaliHLqVK7k23movzmxCYm0PtmNDFwq4vnefnf8zCW1MvPpQtWLceuCzBNkdrouHEIGPlGrTaA=='}}]


In [7]:
print(response["messages"][-1].response_metadata["model_name"])

gemini-3.1-flash-lite
